# SETUP ENVIRONMENT

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
!nvidia-smi   # <-- run this first — tell me which GPU you gots

%cd /content/drive/MyDrive/SpaceAI/UHI2-main

Fri Mar 20 20:57:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   66C    P8             18W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import psutil

ram_gb = psutil.virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

Your runtime has 56.9 gigabytes of available RAM

You are using a high-RAM runtime!


In [3]:
!pip install -r /content/drive/MyDrive/SpaceAI/UHI2-main/competition/requirements.txt --quiet

In [ ]:
# Force reinstall smp if version mismatch suspected (common issue)
!pip install --upgrade segmentation-models-pytorch --quiet

import torch
import segmentation_models_pytorch as smp

print("PyTorch:", torch.__version__)
print("SMP version:", smp.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3 :.1f} GB")

torch.cuda.empty_cache()

PyTorch: 2.10.0+cu128
SMP version: 0.5.0
GPU: NVIDIA L4
VRAM: 22.0 GB


In [ ]:

# Rough estimate — run this before full training
model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1
).cuda().eval()

x = torch.randn(8, 3, 512, 512).cuda()   # simulate your batch

with torch.no_grad(), torch.amp.autocast('cuda'):
    y = model(x)

peak = torch.cuda.max_memory_allocated() / 1024**3
print(f"Forward pass peak memory (batch=8): {peak:.2f} GB")

del model, x, y
torch.cuda.empty_cache()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

Forward pass peak memory (batch=8): 1.05 GB


# TRAINING MODEL

In [4]:
# Run this before !python -m competition.train
import torch
torch.backends.cudnn.benchmark = True   # lets cuDNN pick fastest kernels
torch.backends.cudnn.deterministic = False

In [7]:
# Train the model
#!python train.py

!python -m competition.train

[Dataset Check] Total training images found: 0
Track 3: Built-Up Area Segmentation - Training
[Device] Using CUDA: NVIDIA L4
[Dataset] Found 27300 image-label pairs
[Data] Train: 23205, Val: 4095
/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/content/drive/MyDrive/SpaceAI/UHI2-main/competition/dataset.py:110: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(5.0, 25.0), p=0.2),
[Model] U-Net with resnet50 encoder loaded

[Training] 30 epochs, batch_size=16, AMP enabled
------------------------------------------------------------
Epoch   1/30 | Train Loss: 0.3093 IoU: 0.5453 | Val Loss: 0.1927 IoU: 0.7227 Dice: 0.7587 mIoU: 0.8291 Kappa: 0.1992 | LR: 0.000150 | 694.2s
  -> Saved best model (mIoU: 0.8291)
Epoch   2/30 | Train Loss: 0.2113 IoU: 0.6566 | Val Lo

In [ ]:
!nvidia-smi --query-gpu=timestamp,name,utilization.gpu,memory.used,memory.total --format=csv -l 3

# MODEL PREDICTION

In [8]:
!python -m competition.predict

[Dataset Check] Total training images found: 0
Track 3: Built-Up Area Segmentation - Inference
[Device] Using CUDA: NVIDIA L4
[Model] Loaded checkpoint from epoch 23
        Val IoU: 0.7620, Val Dice: 0.7958, Val mIoU: 0.8551, Val Kappa: 0.2228
[Dataset] Found 216 test images

[Inference] Processing 216 test images...
            Threshold: 0.3
  Processed 50/216
  Processed 100/216
  Processed 150/216
  Processed 200/216
  Processed 216/216

[Verify] Checking 216 output masks...
[Verify] All masks are 512x512 with values in {0, 1} [OK]
[Verify] Images with built-up detections: 54/216
[Info] Visual previews saved to: /content/drive/MyDrive/SpaceAI/UHI2-main/competition/output/visual_preview

[Done] Submission ZIP: /content/drive/MyDrive/SpaceAI/UHI2-main/competition/output/submission.zip
       Size: 0.12 MB
       Contains: 216 PNG masks
